# Module 9 · 圖像 3：現代影像表示（ViT 與 CLIP）

> **本節定位｜2026 主流核心**
> 過去用 VGG16（TensorFlow/Keras）做轉移學習抽特徵；**2026 主流已換成
> Vision Transformer (ViT) 與 CLIP**，且生態系是 **PyTorch + HuggingFace / timm**。
> 本節用預訓練 ViT/CLIP 把影像 `(N,C,H,W)` 變成通用特徵向量。

## 學習目標
- 用 `timm` 載入預訓練 **ViT** 抽取影像特徵向量（取代舊的 VGG16 流程）。
- 理解 ViT 的核心：把影像切成 **patch**，當成「視覺 token」序列丟進 Transformer。
- 用 **CLIP** 得到「**和文字對齊**」的影像嵌入，並做 **zero-shot 分類**。

In [ ]:
import os, urllib.request
import numpy as np
from PIL import Image

def load_taiwan_image():
    """真實的台北 101 照片（Wikimedia Commons, CC BY-SA；下載一次後快取）。"""
    url = ("https://upload.wikimedia.org/wikipedia/commons/thumb/"
           "2/2d/Taipei_Taiwan_Taipei-101-Tower-01.jpg/960px-Taipei_Taiwan_Taipei-101-Tower-01.jpg")
    cache = os.path.join(os.path.expanduser("~/.cache"), "taipei101.jpg")
    if not os.path.exists(cache):
        req = urllib.request.Request(url, headers={"User-Agent": "iSpan-DM-course/1.0 (educational)"})
        data = urllib.request.urlopen(req, timeout=60).read()
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        open(cache, "wb").write(data)
    return Image.open(cache).convert("RGB")

sample = load_taiwan_image()                  # 真實台灣地標照片：台北 101
print(f"真實影像：台北 101（Taipei 101, 台灣），尺寸 {sample.size}")

try:
    import torch
    HAS = True
except Exception:
    HAS = False
    print("未安裝 torch，請 `uv sync --extra multimodal`。說明仍可閱讀。")

## 1. ViT 的核心直覺：影像 = patch 序列

ViT 把 224×224 影像切成 16×16 的 patch（共 14×14=196 個），每個 patch 線性投影成一個向量
（= 一個「視覺 token」），再加上位置編碼丟進標準 Transformer。
**影像就這樣被當成「token 序列」**，和文字 Transformer 同一套架構。

- 輸入 `(N, 3, 224, 224)` → patch 化 → `(N, 196+1, D)`（+1 是 `[CLS]`）→ 取 `[CLS]` 當影像特徵。

## 2. 用 timm 載入預訓練 ViT 抽特徵

`timm` 是 2026 影像模型的事實標準庫，收錄上千個預訓練模型。
設 `num_classes=0` 即可把模型當成「特徵抽取器」。

In [ ]:
if HAS:
    try:
        import timm
        model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
        model.eval()

        # 用模型自帶的正確前處理
        cfg = timm.data.resolve_data_config({}, model=model)
        transform = timm.data.create_transform(**cfg)
        x = transform(sample).unsqueeze(0)        # (1, 3, 224, 224)
        with torch.no_grad():
            feat = model(x)                       # (1, D)
        print(f"輸入: {tuple(x.shape)}  (N, C, H, W)")
        print(f"ViT 影像特徵: {tuple(feat.shape)}  (N, D)  ← 通用特徵向量，可接分類器/檢索")
    except Exception as e:
        print("（未能下載 ViT 權重，略過）", e)

> **對照舊教材**：先前用 `tensorflow.keras.applications.VGG16` 取 512 維特徵。
> 現在改用 PyTorch + timm 的 ViT，理由：(1) 2026 生態系以 PyTorch 為主；
> (2) ViT/Transformer 是現代視覺骨幹；(3) timm 一行切換上千種模型。

## 3. CLIP：和文字對齊的影像嵌入 + Zero-shot 分類

CLIP 用 4 億組「圖–文」配對訓練，把影像與文字嵌到**同一個向量空間**。
因此可以「不訓練」就分類：把候選類別寫成文字，看哪個和影像最相近。

In [ ]:
if HAS:
    try:
        from transformers import CLIPModel, CLIPProcessor
        clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        cproc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

        candidate_labels = ["a photo of a building", "a photo of a flower", "a photo of a dog", "a photo of a car"]
        inputs = cproc(text=candidate_labels, images=sample, return_tensors="pt", padding=True)
        with torch.no_grad():
            out = clip(**inputs)
        probs = out.logits_per_image.softmax(dim=1)[0]   # (num_labels,)
        print("CLIP zero-shot 分類（真實台北 101 照片）：")
        for label, p in zip(candidate_labels, probs):
            print(f"  {p.item():.3f}  {label}")
        print("→ 預期 'building' 機率最高（CLIP 正確辨識真實地標照片內容）。")
        print("\nimage_embeds 形狀:", tuple(out.image_embeds.shape), " (N, D) 與文字同空間")
    except Exception as e:
        print("（未能下載 CLIP 權重，略過）", e)

## 小結

| 方法 | 取得 | 特性 | 2026 定位 |
|:--|:--|:--|:--|
| VGG16（舊） | Keras 中間層 | CNN 特徵 | 已被取代 |
| **ViT (timm)** | `num_classes=0` | patch 序列 + Transformer | 現代視覺骨幹 |
| **CLIP** | 圖文對齊嵌入 | 可 zero-shot、跨模態 | 多模態與檢索基礎 |

**下一節 `04_augmentation_and_datasets`**：訓練視覺模型還需要**資料增強**與
**大規模資料集的組織方式**（ImageFolder / HF datasets / WebDataset）。